<a href="https://colab.research.google.com/github/DreWona/Module3_CameraSimilarity/blob/main/Module_3_Camera_Sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
%matplotlib inline
import pandas as pd

import json

In [109]:
camera_feature_map = {}

In [110]:
with open('cam_deet.json', 'r') as in_file:
  for this_camera in json.load(in_file):
    #numerical feature points
    mp = this_camera.get('Effective megapixels')
    crop = this_camera.get('Crop factor')
    screen = this_camera.get('Screen size')

    if any(v is None or v == '' for v in [mp, crop, screen]):
      continue
      #Remap feature yes or no feature points to 1 or 0
      #dataset id  as brand+camera model because the aarchive didnt come with a ID's
    camera_id = str(this_camera['Brand']).strip() + ' ' + str(this_camera['Model']).strip()
    camera_feature_map[camera_id] = {
        'megapixels': float(str(mp)),
        'crop_factor': float(str(crop)),
        'screen_size': float(str(screen).replace(',', '.')),
        'raw_support': 1 if str(this_camera.get('RAW support', '')).lower() == 'yes' else 0,
        'wireless': 1 if str(this_camera.get('Wireless', '')).lower() == 'yes' else 0,
        'aperture_priority': 1 if str(this_camera.get('Aperture priority', '')).lower() == 'yes' else 0
    }
  print(f"Cameras loaded: {len(camera_feature_map)}")


Cameras loaded: 1841


In [111]:
df = pd.DataFrame.from_dict(camera_feature_map, orient='index').fillna(0)
df.head(1167)

,megapixels,crop_factor,screen_size,raw_support,wireless,aperture_priority
Canon EOS 850D,24.1,1.61,3.0,1,1,1
Canon EOS-1D X Mark III,20.1,1.00,3.2,1,1,1
Canon EOS M200,24.1,1.61,3.0,1,1,1
Canon EOS 90D,32.5,1.61,3.0,1,1,1
Canon EOS M6 Mark II,32.5,1.61,3.0,1,1,1
...,...,...,...,...,...,...
Pentax Optio S7,7.1,6.02,2.5,0,0,0
Pentax Optio A10,8.0,4.87,2.5,0,0,0
Pentax *ist DL2,6.1,1.53,2.5,0,0,1
Pentax K110D,6.1,1.53,2.5,0,0,1


In [112]:
df.index[[270, 500, 800, 750, 400]]

Index(['Fujifilm X-E2', 'Nikon Coolpix S2700', 'Olympus FE-115',
       'Olympus PEN E-P1', 'Leica D-LUX 4'],
      dtype='object')

In [113]:
df[df.index.str.contains('Nikon')][:5]

,megapixels,crop_factor,screen_size,raw_support,wireless,aperture_priority
Nikon Z50,20.9,1.53,3.2,1,1,1
Nikon Coolpix W150,13.2,7.21,2.7,0,1,0
Nikon Coolpix B600,16.0,5.62,3.0,0,1,0
Nikon Coolpix A1000,16.0,5.62,3.0,1,1,1
Nikon Z7,45.7,1.00,3.2,1,1,1


In [115]:
df[df.index.str.contains('Sony')][:5]

,megapixels,crop_factor,screen_size,raw_support,wireless,aperture_priority
Sony Alpha a9 II,24.2,1.01,3.0,1,1,1
Sony a6600,24.2,1.53,3.0,1,1,1
Sony a6100,24.2,1.53,3.0,1,1,1
Sony a7R IV,61.0,1.01,3.0,1,1,1
Sony Alpha a6400,24.2,1.53,3.0,1,1,1


In [116]:
from sklearn.metrics.pairwise import cosine_distances
from sklearn.preprocessing import MinMaxScaler

In [117]:
num_cols = ['megapixels', 'crop_factor', 'screen_size']
df_norm = df.copy()
df_norm[num_cols] = MinMaxScaler().fit_transform(df[num_cols])

In [127]:
#My 3 cameras sets
query_cameras = ['Canon EOS 850D', 'Sony a7R IV', 'Nikon Z7']

In [131]:
# 10 Reps of similar cams
for target_camera_id in query_cameras:

    target_camera_ratings = df_norm.loc[target_camera_id]
    distances = cosine_distances(df_norm, [target_camera_ratings])[:, 0]
    query_distances = list(zip(df_norm.index, distances))

    print(f"\n{'='*55}")
    print(f"  Top 10 most similar to: {target_camera_id}")
    print(f"{'='*55}")

    count = 0
    for similar_camera_id, similar_score in sorted(query_distances, key=lambda x: x[1]):
        if similar_camera_id == target_camera_id:
            continue
        print(similar_camera_id, round(1 - similar_score, 4))
        count += 1
        if count == 10:
            break


  Top 10 most similar to: Canon EOS 850D
Canon EOS M200 1.0
Canon EOS 250D 1.0
Canon EOS M50 1.0
Canon EOS Rebel T7 1.0
Canon PowerShot G1 X Mark III 1.0
Canon EOS M100 1.0
Canon EOS Rebel SL2 1.0
Canon EOS Rebel T7i 1.0
Canon EOS 77D 1.0
Canon EOS M6 1.0

  Top 10 most similar to: Sony a7R IV
Leica Q2 0.9949
Panasonic Lumix DC-S1R 0.9941
Nikon Z7 0.9928
Nikon D850 0.9928
Sony Alpha a7R III 0.9903
Sony Alpha A99 II 0.9903
Sony Cyber-shot DSC-RX1R II 0.9903
Sony Alpha 7R II 0.9903
Pentax K-1 Mark II 0.9813
Pentax K-1 0.9813

  Top 10 most similar to: Nikon Z7
Nikon D850 1.0
Panasonic Lumix DC-S1R 0.9999
Leica Q2 0.9994
Sony Alpha a7R III 0.9993
Sony Alpha A99 II 0.9993
Sony Cyber-shot DSC-RX1R II 0.9993
Sony Alpha 7R II 0.9993
Pentax K-1 Mark II 0.9972
Pentax K-1 0.9972
Canon EOS 90D 0.993


In [134]:

all_results = []

for target_camera_id in query_cameras:
    target_camera_ratings = df_norm.loc[target_camera_id]
    distances = cosine_distances(df_norm, [target_camera_ratings])[:, 0]
    query_distances = list(zip(df_norm.index, distances))

    count = 0
    for similar_camera_id, similar_score in sorted(query_distances, key=lambda x: x[1]):
        if similar_camera_id == target_camera_id:
            continue
        all_results.append({
            'Cameras': target_camera_id,
            'similar_camera': similar_camera_id,
            'cosine_similarity': round(1 - similar_score, 4)
        })
        count += 1
        if count == 10:
            break


results_df = pd.DataFrame(all_results)
results_df.to_csv('Similar_Cameras.csv', index=False)